# Dataset 3 — Centrality Model Selection (p = 0)

Same selection logic as Dataset 1, applied to the static Dataset 3. One row per bank (no temporal axis), so `ModelTrainer` uses `split='random'` and top-1% cohort scoring is skipped. Best model saved to `models/dataset_3/04_a`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'src').exists() and (path / 'requirements.txt').exists():
            return path
    raise FileNotFoundError('Project root not found.')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
from src.models.ml_train_and_store import (
    ModelTrainer,
    CLASSICAL_FEATURE_CANDIDATES,
    load_gnn_dataset,
    load_model,
    make_pipeline,
)

pd.set_option('display.max_columns', 200)
TARGET_COL = 'log_systemic_risk_label'
print('Project root:', PROJECT_ROOT)

Project root: /Users/rubenmarques/Documents/Repositórios/Thesis


## Load Dataset 3 (classical / centrality features + baseline target)

In [2]:
FEATURES_PATH = PROJECT_ROOT / 'src' / 'data' / 'classical_features' / 'dataset_3' / 'classical_features_dataset3.parquet'
TARGET_PATH   = PROJECT_ROOT / 'src' / 'datasets' / 'dataset_3' / 'targets' / 'target.csv'  # p = 0 baseline

features = pd.read_parquet(FEATURES_PATH)
targets  = pd.read_csv(TARGET_PATH)
df = features.merge(targets, on='bank_id', how='inner')
feature_cols = [c for c in CLASSICAL_FEATURE_CANDIDATES if c in df.columns]
df = df.dropna(subset=[TARGET_COL]).reset_index(drop=True)
df.shape, feature_cols

((10000, 11),
 ['degree_centrality_total',
  'weighted_degree_total',
  'betweenness_centrality',
  'closeness_centrality',
  'eigenvector_centrality',
  'pagerank'])

In [3]:
trainer = ModelTrainer(df=df, feature_cols=feature_cols, target_col=TARGET_COL, split='random')
trainer.train_df.shape, trainer.val_df.shape, trainer.test_df.shape

((7000, 11), (1500, 11), (1500, 11))

## Define Models

In [4]:
DISPLAY_COLS = ["model", "train_mae", "validation_mae", "train_rmse", "validation_rmse"]

candidate_models = {
    "Linear Regression": make_pipeline(LinearRegression()),
    "Ridge":             make_pipeline(Ridge(alpha=1.0)),
    "MLP":               make_pipeline(MLPRegressor(hidden_layer_sizes=(32,63,16), max_iter=300, activation="relu", learning_rate="adaptive", learning_rate_init=0.001, early_stopping=True, n_iter_no_change=3, random_state=42)),
    "Random Forest":     make_pipeline(RandomForestRegressor(n_estimators=100, random_state=42)),
    "Gradient Boosting": make_pipeline(HistGradientBoostingRegressor(max_iter=200, random_state=42)),
    "XGBoost":           make_pipeline(XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)),
}
list(candidate_models)

['Linear Regression',
 'Ridge',
 'MLP',
 'Random Forest',
 'Gradient Boosting',
 'XGBoost']

## Train

In [5]:
trainer.train_all(candidate_models)
trainer.leaderboard().assign(**{'val/train_rmse': lambda d: (d['validation_rmse'] / d['train_rmse']).round(2), 'val/train_mae': lambda d: (d['validation_mae'] / d['train_mae']).round(2)})[DISPLAY_COLS + ['val/train_rmse', 'val/train_mae']]

,model,train_mae,validation_mae,train_rmse,validation_rmse,val/train_rmse
0,XGBoost,0.100,0.135,0.245,0.315,1.29
1,Gradient Boosting,0.101,0.134,0.249,0.324,1.30
2,Random Forest,0.060,0.152,0.144,0.367,2.55
3,Linear Regression,0.335,0.327,0.542,0.537,0.99
4,MLP,0.532,0.528,0.766,0.758,0.99
5,Ridge,0.609,0.603,0.847,0.834,0.98


## Hyperparameter Tuning

In [6]:
def tune(trainer, base_model, param_distributions, name, n_iter=40):
    X = pd.concat([trainer.train_df[trainer.feature_cols], trainer.val_df[trainer.feature_cols]])
    y = pd.concat([trainer.train_df[trainer.target_col],   trainer.val_df[trainer.target_col]])
    split_idx = np.concatenate([np.full(len(trainer.train_df), -1), np.zeros(len(trainer.val_df), dtype=int)])
    search = RandomizedSearchCV(base_model, param_distributions, n_iter=n_iter,
                                cv=PredefinedSplit(split_idx), scoring='neg_root_mean_squared_error',
                                random_state=42, n_jobs=-1)
    search.fit(X, y)
    trainer.train(search.best_estimator_, name=name)
    trainer.best_params[name] = search.best_params_
    return search.best_params_

RF_PARAMS = {
    'model__n_estimators': [100, 200, 300, 400, 500, 600],
    'model__max_depth': [None, 5, 10, 15, 20, 30],
    'model__min_samples_leaf': [1, 2, 5, 10, 15, 20],
    'model__min_samples_split': [2, 5, 10, 15, 20],
    'model__max_features': ['sqrt', 'log2', 0.5, 0.8, 1.0],
}
GB_PARAMS = {
    'model__max_iter': [100, 200, 300, 400, 500, 600],
    'model__max_depth': [3, 4, 5, 6, 8, None],
    'model__learning_rate': [0.005, 0.01, 0.05, 0.1, 0.2, 0.3],
    'model__min_samples_leaf': [5, 10, 20, 50, 100],
    'model__l2_regularization': [1e-4, 1e-3, 1e-2, 0.1, 1.0],
    'model__max_leaf_nodes': [15, 20, 30, 40, 50, 60],
    'model__max_bins': [64, 128, 255],
}
XGB_PARAMS = {
    'model__n_estimators': [100, 200, 400, 600, 800],
    'model__max_depth': [3, 4, 5, 6, 8, 10],
    'model__learning_rate': [0.005, 0.01, 0.05, 0.1, 0.2, 0.3],
    'model__subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.5, 0.6, 0.7, 0.8, 1.0],
    'model__min_child_weight': [1, 2, 5, 10],
    'model__gamma': [0, 0.1, 0.5, 1.0, 2.0],
    'model__reg_alpha': [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1.0],
    'model__reg_lambda': [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1.0, 2.0],
}
MLP_PARAMS = {
    'model__hidden_layer_sizes': [(64,), (128,), (256,), (128, 64), (256, 128), (128, 64, 32), (256, 128, 64)],
    'model__activation': ['relu', 'tanh'],
    'model__alpha': [1e-5, 1e-4, 1e-3, 1e-2, 0.1],
    'model__learning_rate_init': [1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 0.05],
    'model__learning_rate': ['constant', 'adaptive'],
    'model__batch_size': [32, 64, 128, 'auto'],
}

In [7]:
tune(trainer, make_pipeline(RandomForestRegressor(random_state=42)),         RF_PARAMS,  'Random Forest (tuned)')
tune(trainer, make_pipeline(HistGradientBoostingRegressor(random_state=42)), GB_PARAMS,  'Gradient Boosting (tuned)')
tune(trainer, make_pipeline(XGBRegressor(random_state=42)),                  XGB_PARAMS, 'XGBoost (tuned)')
tune(trainer, make_pipeline(MLPRegressor(max_iter=500, early_stopping=True, n_iter_no_change=20, random_state=42)), MLP_PARAMS, 'MLP (tuned)')

/Users/rubenmarques/Documents/Repositórios/Thesis/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/rubenmarques/Documents/Repositórios/Thesis/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/rubenmarques/Documents/Repositórios/Thesis/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/rubenmarques/Documents/Repositórios/Thesis/venv/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimize

{'model__learning_rate_init': 0.0005,
 'model__learning_rate': 'adaptive',
 'model__hidden_layer_sizes': (256, 128),
 'model__batch_size': 64,
 'model__alpha': 1e-05,
 'model__activation': 'relu'}

In [8]:
trainer.leaderboard().assign(**{'val/train_rmse': lambda d: (d['validation_rmse'] / d['train_rmse']).round(2), 'val/train_mae': lambda d: (d['validation_mae'] / d['train_mae']).round(2)})[DISPLAY_COLS + ['val/train_rmse', 'val/train_mae']]

,model,train_mae,validation_mae,train_rmse,validation_rmse,val/train_rmse
0,XGBoost (tuned),0.120,0.140,0.270,0.311,1.15
1,Gradient Boosting (tuned),0.117,0.135,0.277,0.315,1.14
2,XGBoost,0.100,0.135,0.245,0.315,1.29
3,Gradient Boosting,0.101,0.134,0.249,0.324,1.30
4,Random Forest (tuned),0.102,0.151,0.230,0.347,1.51
5,Random Forest,0.060,0.152,0.144,0.367,2.55
6,MLP (tuned),0.265,0.260,0.483,0.466,0.96
7,Linear Regression,0.335,0.327,0.542,0.537,0.99
8,MLP,0.532,0.528,0.766,0.758,0.99
9,Ridge,0.609,0.603,0.847,0.834,0.98


## Save best model

In [9]:
SAVE_DIR = PROJECT_ROOT / 'src' / 'models' / 'dataset_3' / '04_a'
best_name = trainer.leaderboard().iloc[0]['model']
print('Best:', best_name)
path = trainer.save_model(best_name, SAVE_DIR)

Best: XGBoost (tuned)
Saved 'XGBoost (tuned)' → /Users/rubenmarques/Documents/Repositórios/Thesis/src/models/dataset_3/04_a/XGBoost_(tuned).joblib
